In [0]:
# =============================================================================
# BLS CEW — Silver Cleaning (VERSION CORRIGÉE FINALE)
# Corrections appliquées :
#   1. Typage explicite des colonnes (inferSchema instable sur 10 ans CSV)
#   2. Filtre nulls critiques avant tout traitement
#   3. sector_type : "Other" exclu, "Total" séparé
#   4. Valeurs négatives : mesures stock seulement (OTY conservés)
#   5. disclosure_code → flag booléen is_disclosed
#   6. Déduplication : clé composite complète (size_code inclus)
#   7. Colonnes dérivées : naics2, covid_period
#   8. Partitionnement : (year, sector_type)
#   9. industry_titles : join robuste avec fallback
#  10. saveAsTable Unity Catalog avec schema explicite
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T
import re

# ── CONFIG ────────────────────────────────────────────────────────────────────
RAW_GLOB    = "/Volumes/bls_cew/raw/bronze_cew/*.csv"
SILVER_PATH = "/Volumes/bls_cew/silver/silver_cew/cleaned_data"
INDUSTRIES  = "/Volumes/bls_cew/default/industry_tites/industry-titles.csv"

WRITE_FORMAT = "delta"
WRITE_MODE   = "overwrite"

# ── HELPER ───────────────────────────────────────────────────────────────────
def normalize_col(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


# =============================================================================
# ÉTAPE 1 — LECTURE BRONZE
# =============================================================================
print("=== Lecture Bronze ===")
df_raw = spark.read.csv(RAW_GLOB, header=True, inferSchema=True)
print(f"Lignes brutes lues : {df_raw.count():,}  |  Colonnes : {len(df_raw.columns)}")

# Normalisation des noms de colonnes
for c in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(c, normalize_col(c))

df = df_raw


# =============================================================================
# ÉTAPE 2 — TYPAGE EXPLICITE
# CORRECTION : inferSchema=True est instable sur 10 fichiers CSV annuels.
# Une cellule "N" (non-divulguée) peut faire inférer "string" sur toute
# une colonne numérique. On caste explicitement après normalisation.
# =============================================================================
print("\n=== Typage explicite ===")

INT_COLS = [
    "year", "agglvl_code", "own_code", "size_code",
]
LONG_COLS = [
    "annual_avg_emplvl", "annual_avg_estabs",
    "total_annual_wages", "taxable_annual_wages",
    "annual_contributions",
]
DOUBLE_COLS = [
    "annual_avg_wkly_wage", "avg_annual_pay",
    "oty_annual_avg_emplvl_chg",    "oty_annual_avg_emplvl_pct_chg",
    "oty_total_annual_wages_chg",   "oty_total_annual_wages_pct_chg",
    "oty_avg_annual_pay_chg",       "oty_avg_annual_pay_pct_chg",
]

for c in INT_COLS:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(T.IntegerType()))
for c in LONG_COLS:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(T.LongType()))
for c in DOUBLE_COLS:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(T.DoubleType()))

# industry_code et area_fips restent STRING pour le join et les agrégations
for c in ["industry_code", "area_fips"]:
    if c in df.columns:
        df = df.withColumn(c, F.trim(F.col(c).cast(T.StringType())))

print("Typage appliqué.")


# =============================================================================
# ÉTAPE 3 — FILTRE NULLS CRITIQUES
# CORRECTION : absent dans la version originale.
# Une ligne sans year/area_fips/industry_code/own_code est inutilisable
# pour tous les KPIs. La conserver pollue les GROUP BY.
# =============================================================================
print("\n=== Filtre nulls critiques ===")

CRITICAL_KEYS = ["year", "area_fips", "industry_code", "own_code"]
for k in CRITICAL_KEYS:
    if k in df.columns:
        before = df.count()
        df = df.filter(F.col(k).isNotNull())
        dropped = before - df.count()
        if dropped > 0:
            print(f"  Lignes supprimées ({k} null) : {dropped:,}")

print(f"Lignes restantes : {df.count():,}")


# =============================================================================
# ÉTAPE 4 — FILTRE agglvl_code = 73
# CONSERVÉ : niveau État + Industrie (NAICS 6 chiffres), meilleur compromis
# granularité / volume pour ce pipeline.
# =============================================================================
print("\n=== Filtre agglvl_code = 73 ===")

if "agglvl_code" not in df.columns:
    raise ValueError("Colonne agglvl_code introuvable après normalisation.")

df = df.filter(F.col("agglvl_code") == 73)
print(f"Lignes après filtre agglvl=73 : {df.count():,}")


# =============================================================================
# ÉTAPE 5 — sector_type
# CORRECTION : la version originale laissait "Other" dans la Silver.
#
# own_code = 0  → "Total"    (ligne de totalisation BLS, gardée pour contrôle)
# own_code = 1  → "Private"
# own_code = 2,3,5 → "Public" (local, état, fédéral)
# own_code = 4,8,9,autre → exclu (< 0.5% des lignes, non classifiable)
#
# Les lignes "_exclude" sont supprimées ici, avant la Silver.
# Les lignes "Total" (own_code=0) sont conservées pour validation croisée
# mais marquées pour être exclues des agrégations KPI en Gold.
# =============================================================================
print("\n=== Création sector_type ===")

if "own_code" not in df.columns:
    raise ValueError("Colonne own_code introuvable.")

df = df.withColumn(
    "sector_type",
    F.when(F.col("own_code") == 0, F.lit("Total"))
     .when(F.col("own_code") == 5, F.lit("Private"))
     .when(F.col("own_code").isin(1, 2, 3), F.lit("Public"))
     .otherwise(F.lit("_exclude"))
)

before_excl = df.count()
df = df.filter(F.col("sector_type") != "_exclude")
print(f"Lignes exclues (own_code non classifié) : {before_excl - df.count():,}")
df.groupBy("sector_type").count().orderBy("sector_type").show()


# =============================================================================
# ÉTAPE 6 — TRAITEMENT DES VALEURS NÉGATIVES
# CORRECTION : version originale trop agressive + liste incomplète.
#
# Règle :
#   - Mesures de STOCK (emploi, salaires, estabs) → NULL si < 0
#     Un emploi ou salaire négatif = erreur de saisie.
#   - Colonnes OTY (variation annuelle) → ON NE TOUCHE PAS
#     oty_*_pct_chg = -6.8 en 2020 = choc COVID légitime.
# =============================================================================
print("\n=== Nettoyage valeurs négatives (stock seulement) ===")

STOCK_MEASURES = [
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "taxable_annual_wages",
    "annual_contributions",
    "annual_avg_wkly_wage",
    "avg_annual_pay",
]
for c in STOCK_MEASURES:
    if c in df.columns:
        neg_count = df.filter(F.col(c) < 0).count()
        if neg_count > 0:
            print(f"  {c} : {neg_count:,} valeurs négatives → NULL")
        df = df.withColumn(c, F.when(F.col(c) < 0, F.lit(None)).otherwise(F.col(c)))

# Les colonnes oty_* NE SONT PAS modifiées — les négatifs sont valides.
print("Colonnes OTY conservées intactes (variations négatives = données valides).")


# =============================================================================
# ÉTAPE 7 — disclosure_code → flag booléen
# CORRECTION : la version originale supprimait la colonne.
# Supprimer disclosure_code fait perdre l'info de confidentialité BLS.
# "N" dans cette colonne = donnée masquée pour protéger les entreprises.
#
# Solution : convertir en is_disclosed (True = donnée publique fiable).
# En Gold, on filtrera sur is_disclosed = True pour les KPIs.
# =============================================================================
print("\n=== Conversion disclosure_code → is_disclosed ===")

DISC_MAP = {
    "disclosure_code":     "is_disclosed",
    "lq_disclosure_code":  "lq_is_disclosed",
    "oty_disclosure_code": "oty_is_disclosed",
}
for src, dst in DISC_MAP.items():
    if src in df.columns:
        masked = df.filter(F.col(src).isNotNull() & (F.col(src) != "")).count()
        print(f"  {src} : {masked:,} lignes masquées par le BLS")
        df = df.withColumn(
            dst,
            F.when(
                F.col(src).isNull() | (F.col(src) == ""),
                F.lit(True)
            ).otherwise(F.lit(False))
        ).drop(src)

print("Flags is_disclosed créés.")


# =============================================================================
# ÉTAPE 8 — COLONNES DÉRIVÉES
# AJOUT : calculées une fois en Silver → évite de les recalculer dans chaque
# requête SQL du dashboard.
# =============================================================================
print("\n=== Colonnes dérivées ===")

# naics2 : 2 premiers chiffres NAICS → secteur large (pour les tendances IA)
if "industry_code" in df.columns:
    df = df.withColumn("naics2", F.substring(F.col("industry_code"), 1, 2))

# covid_period : flag macroéconomique
if "year" in df.columns:
    df = df.withColumn(
        "covid_period",
        F.when(F.col("year") < 2020,  F.lit("pre_covid"))
         .when(F.col("year") == 2020, F.lit("choc"))
         .when(F.col("year") == 2021, F.lit("reprise"))
         .otherwise(F.lit("post_covid"))
    )

print("Colonnes naics2 et covid_period ajoutées.")


# =============================================================================
# ÉTAPE 9 — DÉDUPLICATION
# CORRECTION : la version originale omettait size_code dans la clé.
# Le BLS émet plusieurs lignes pour le même (year, area, industry, own)
# selon la taille d'établissement (size_code). Sans size_code dans la clé,
# des lignes distinctes étaient fusionnées à tort.
# =============================================================================
print("\n=== Déduplication ===")

KEY_COLS = [c for c in [
    "year", "area_fips", "industry_code",
    "own_code", "agglvl_code", "size_code"
] if c in df.columns]

before_dedup = df.count()
df = df.dropDuplicates(KEY_COLS)
print(f"Doublons supprimés : {before_dedup - df.count():,}")
print(f"Lignes après dédup : {df.count():,}")


# =============================================================================
# ÉTAPE 10 — JOIN INDUSTRY TITLES
# CORRECTION : plusieurs problèmes dans la version originale :
#   a) Pas de vérification que le fichier existe et est lisible
#   b) Pas de contrôle du taux de correspondance après le join
#   c) industry_code doit être STRING des deux côtés avant le join
#   d) Le join "left" est correct — on ne veut pas perdre de lignes
#      si un code NAICS n'a pas de libellé dans le fichier de référence.
# =============================================================================
print("\n=== Join industry titles ===")

try:
    df_ind = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(INDUSTRIES)
    )
    print(f"Fichier industry titles lu : {df_ind.count()} codes")

    # Normalisation des colonnes du fichier de référence
    for c in df_ind.columns:
        df_ind = df_ind.withColumnRenamed(c, normalize_col(c))

    print("Colonnes industry titles :", df_ind.columns)

    # Vérification que les colonnes attendues existent
    # Le BLS nomme ces colonnes "industry_code" et "industry_title"
    if "industry_code" not in df_ind.columns:
        raise ValueError(f"Colonne 'industry_code' introuvable. Colonnes disponibles : {df_ind.columns}")
    if "industry_title" not in df_ind.columns:
        raise ValueError(f"Colonne 'industry_title' introuvable. Colonnes disponibles : {df_ind.columns}")

    # Préparer le référentiel : code STRING nettoyé + dédupliqué
    df_ind = (
        df_ind
        .select(
            F.trim(F.col("industry_code").cast(T.StringType())).alias("industry_code"),
            F.trim(F.col("industry_title")).alias("industry_name"),
        )
        .dropDuplicates(["industry_code"])
    )

    # S'assurer que industry_code est STRING et nettoyé dans df également
    # (déjà fait à l'étape 2, mais on s'assure de la cohérence)
    df = df.withColumn("industry_code", F.trim(F.col("industry_code").cast(T.StringType())))

    # Join left : on conserve toutes les lignes de df même sans libellé
    before_join = df.count()
    df = df.join(df_ind, on="industry_code", how="left")

    # Contrôle qualité du join
    matched   = df.filter(F.col("industry_name").isNotNull()).count()
    unmatched = df.filter(F.col("industry_name").isNull()).count()
    match_rate = matched / before_join * 100

    print(f"  Lignes avec industry_name    : {matched:,}  ({match_rate:.1f}%)")
    print(f"  Lignes sans correspondance   : {unmatched:,}")

    # Alerte si le taux de correspondance est trop faible
    if match_rate < 80:
        print(f"  ⚠ ALERTE : taux de correspondance faible ({match_rate:.1f}%)")
        print("    Vérifier que les codes NAICS du CSV correspondent au format de la Silver.")
        # Afficher un échantillon des codes non matchés pour diagnostic
        (df.filter(F.col("industry_name").isNull())
           .select("industry_code")
           .distinct()
           .limit(10)
           .show(truncate=False))
    else:
        print(f"  Join OK ({match_rate:.1f}% de correspondance)")

except Exception as e:
    # Si le fichier industry_titles est absent ou corrompu, on continue
    # sans le join plutôt que de planter toute la pipeline.
    print(f"  ⚠ Join industry titles ignoré : {e}")
    print("    La Silver sera créée sans la colonne industry_name.")
    if "industry_name" not in df.columns:
        df = df.withColumn("industry_name", F.lit(None).cast(T.StringType()))


# =============================================================================
# ÉTAPE 11 — VALIDATION FINALE
# =============================================================================
print("\n=== Validation finale ===")

total_rows = df.count()
print(f"Lignes finales   : {total_rows:,}")
print(f"Colonnes         : {len(df.columns)}")
print(f"Colonnes liste   : {sorted(df.columns)}")

print("\n--- Répartition sector_type ---")
df.groupBy("sector_type").count().orderBy("sector_type").show()

print("\n--- Répartition par année ---")
df.groupBy("year").count().orderBy("year").show(15, truncate=False)

print("\n--- Nulls sur colonnes critiques ---")
null_check_cols = [
    c for c in ["year", "area_fips", "industry_code", "own_code",
                "annual_avg_emplvl", "avg_annual_pay", "industry_name"]
    if c in df.columns
]
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in null_check_cols
]).show()

print("\n--- Flags disclosure ---")
disc_flags = [c for c in df.columns if "is_disclosed" in c]
if disc_flags:
    df.select([
        F.sum(F.when(~F.col(c), 1).otherwise(0)).alias(c)
        for c in disc_flags
    ]).show()


# =============================================================================
# ÉTAPE 12 — ÉCRITURE SILVER (Unity Catalog)
# CORRECTION :
#   a) Partitionnement (year, sector_type) au lieu de year seul.
#      Les requêtes dashboard filtrent souvent les deux → lecture directe
#      de la partition sans scan complet de l'année.
#   b) overwriteSchema=True pour permettre les évolutions de schéma.
#   c) saveAsTable pour Unity Catalog (bls_cew.silver.cleaned_data).
# =============================================================================
print("\n=== Écriture Silver Delta (Unity Catalog) ===")

spark.sql("CREATE SCHEMA IF NOT EXISTS bls_cew.silver")

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .partitionBy("year", "sector_type")
      .saveAsTable("bls_cew.silver.cleaned_data")
)

print("Table créée : bls_cew.silver.cleaned_data")
print("Partitions  : year + sector_type")
print("\nSilver prête pour le dashboard et la Gold layer.")